# 04a · Theme 1 write-up: Awareness and Engagement

Synthesis of `02` (strategy) and `03` (tiering, screening, ablation, SHAP). Nike and Adidas TikTok sample (~4.4k videos). Descriptive and predictive screening results; not causal.


## 1. Bottom line

Pre-publish features add little to point-forecasting WER (~3% MAE improvement over brand median). The same signals are useful for ranking candidates with higher top-decile odds (Lift@10 ≈ 2.39×). Strategy remains grounded in `02`; the model is a screening layer.


## 2. Analysis layers

| Layer | Source | Role |
|-------|--------|------|
| Strategy | `02` | Within-brand and between-brand content performance |
| Tiering | `03` `qclass_4` | Coarse Low/Mid/High/Top segmentation |
| Screening | `03` Top-10% CatBoost + CLIP | Shortlist enrichment (Lift@K) |
| Drivers | `03` SHAP | Features used by the screener (associative) |


## 3. Strategy (`02`)

- Lifestyle / OOTD is the largest and relatively strong overall direction.
- Adidas leans lifestyle / GRWM / OOTD; Nike leans technical / performance / promo / sports-action.
- Adidas median WER exceeds Nike in this sample (Mann–Whitney significant).
- Trend, GRWM/OOTD, and collaboration tend to outperform hard promo (`product_promo`, `official_campaign`).
- Within-brand: Adidas Lifestyle and Vibe/OOTD; Nike Technical and Tutorial/Utility.
- Collaboration is above each brand’s own baseline.


## 4. Predictive layer (`03`): locked numbers

### 4.1 Why not exact WER regression

| | WER MAE | Notes |
|-|---------|-------|
| Brand-median baseline | ≈ 0.00493 | |
| CatBoost regression | ≈ 0.00479 | ~3% improvement; not useful as a forecaster |

Better use of the same signals: prioritize unusual high performers.

### 4.2 Tiering: `qclass_4`

- Four train-fold quantile classes; score = expected class $\sum i\,P(Y=i)$.
- Acc ≈ 0.33 (vs ~0.25 random); Q4 OvR AUC ≈ 0.63-0.64; Spearman ≈ 0.23.
- Role: library segmentation, not the headline KPI.

### 4.3 Screening: Top-10% (primary) / Top-25% (robustness)

Author `GroupKFold`; labels from train-fold Q90 / Q75 only. CatBoost tuned on Top-25, then reused for Top-10.

| Model | Lift@5 | Lift@10 | Lift@20 | AUC | Prec@10 |
|-------|--------|---------|---------|-----|----------|
| Top-10% (primary) | 3.10× | 2.39× | 2.06× | 0.686 | 0.241 |
| Top-25% (robustness) | 2.41× | 2.33× | 1.85× | 0.676 | 0.566 |

Among model-ranked top 10%, true top-decile rate is about 24% vs ~10% baseline (2.39× enrichment).

### 4.4 Modality ablation (clean ladder, default CatBoost)

| Step | Add | Lift@10 |
|------|-----|--------|
| M0 | Metadata | 1.97 |
| M1 | + engineered text | 2.32 |
| M2 | + SBERT PCA | 2.12 |
| M3a | + visual format/setting axes | 2.03 |
| M3b | + CLIP visual PCA | 2.59 |
| M4 | + alignment | 2.35 |

Engineered text and CLIP PCA earn Lift. Format/setting axes and alignment help less on Lift (still useful for interpretation and for lining up with `02`).


## 5. Screening drivers (SHAP)

Importance order (pooled): CLIP visual + text embeddings; creator audience/identity; caption traits; duration and timing; visual format/setting and visual–text alignment. Brand-conditional signed SHAP (`03` §10c) is required for direction vs `02`; pooled ranks are not a content playbook.


## 6. Use

1. Select territories from `02`.
2. Score candidates with pre-publish features.
3. Use `qclass_4` for coarse tiers and Top-10% score for shortlist priority.
4. Human review of the shortlist.

**Limits.** *n* ≈ 4.4k; author-grouped CV; CLIP coverage ~84%; sparse taxonomy axes. No virality or causal-lift claim.


## 7. Sources

| Need | Notebook |
|------|----------|
| Strategy tables | `02_descriptive_crosstabs.ipynb` |
| Modeling detail | `03_engagement_modeling.ipynb` |
